# FleetTrust — train the phase models

Produces the two artifacts the app loads:

| file | question | estimators |
|---|---|---|
| `phase_classifier.pkl` | which phase is this site in? | 1 `XGBClassifier` |
| `phase_end.pkl` | when does that phase end? | 3 `XGBRegressor` (P10/P50/P90) |

**Upload these three files first** (folder icon on the left, drag them in). Generate them with `python scripts/export_training_data.py`:

```
data/training/phase_panel.csv
data/training/site_phases.csv
data/training/train_meta.json
```

Nothing else is needed — no repo, no clone. Features are already computed in the panel.

Run the cells top to bottom. Takes about a minute.

## 1 · Install

In [ ]:
!pip install -q xgboost joblib pandas numpy

## 2 · Load

The fingerprint identifies the dataset. The app refuses any model whose fingerprint does not match the data it is serving — that is what stops a stale `.pkl` from quietly making predictions for a dataset that no longer exists.

In [ ]:
import json, math
import numpy as np
import pandas as pd

META = json.load(open('train_meta.json'))
panel = pd.read_csv('phase_panel.csv')
site_phases = pd.read_csv('site_phases.csv')

SEED = META['seed']
CLF_FEATURES = META['classifier_features']
DUR_FEATURES = META['duration_features']
QUANTILES = META['quantiles']
LEVEL = META['interval_level']
N_FOLDS = META['n_folds']
PHASES = META['phase_names']

print('fingerprint ', META['fingerprint'])
print('panel       ', f"{len(panel):,} rows")
print('windows     ', len(site_phases))
print('features    ', len(CLF_FEATURES), 'classifier /', len(DUR_FEATURES), 'duration')

## 3 · Folds and calibration

**Splits hold out whole phase windows, never individual weeks.** Weeks inside one window are near-duplicates — same site, same machines. Split at random and week 3 lands in train while week 4 lands in test; the model recognises the site rather than the pattern and the score is a fiction.

`conformal_pad` widens the quantile band so it covers what it claims. Fitted quantiles on a few hundred rows collapse toward the median — a nominal 80% band measured about 40%.

In [ ]:
def window_folds(df, n_folds=N_FOLDS):
    """Grouped K-fold over whole phase windows. Seeded, so folds never move."""
    windows = sorted(df['window_id'].unique())
    if len(windows) < n_folds:
        return []
    rng = np.random.default_rng(SEED)
    shuffled = list(windows)
    rng.shuffle(shuffled)
    assignment = {w: i % n_folds for i, w in enumerate(shuffled)}
    fold_of = df['window_id'].map(assignment)
    return [(df[fold_of != k], df[fold_of == k]) for k in range(n_folds)]


def conformal_pad(truth, low, high, level=LEVEL):
    """Romano, Patterson & Candes (2019), with the finite-sample correction."""
    if len(truth) == 0:
        return 0.0
    scores = np.maximum(low - truth, truth - high)
    n = len(scores)
    rank = min(int(math.ceil((n + 1) * level)), n)
    return max(float(np.sort(scores)[rank - 1]), 0.0)

print('ok')

## 4 · Phase classifier

Shallow and small on purpose: ~1,200 rows and 20 features. Deeper trees memorise individual site-weeks and the held-out score collapses. `n_jobs=1` for determinism.

In [ ]:
from xgboost import XGBClassifier

def new_classifier():
    return XGBClassifier(
        n_estimators=220, max_depth=4, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9, min_child_weight=4,
        objective='multi:softprob', num_class=len(PHASES),
        tree_method='hist', n_jobs=1, random_state=SEED, verbosity=0,
    )

clf_data = panel.dropna(subset=CLF_FEATURES)
encode = {name: i for i, name in enumerate(PHASES)}

# --- out-of-fold score: every window gets exactly one honest prediction ---
pred, truth = [], []
for train, test in window_folds(clf_data):
    if train.empty or test.empty:
        continue
    m = new_classifier()
    m.fit(train[CLF_FEATURES].to_numpy(float), train['phase'].map(encode).to_numpy(int))
    pred.extend(PHASES[i] for i in m.predict(test[CLF_FEATURES].to_numpy(float)))
    truth.extend(test['phase'])

pred, truth = np.array(pred), np.array(truth)
accuracy = float((pred == truth).mean())
order = {p: i for i, p in enumerate(PHASES)}
within_one = float(np.mean([abs(order[a] - order[b]) <= 1 for a, b in zip(pred, truth)]))
per_phase = {p: float((pred[truth == p] == p).mean()) for p in PHASES if (truth == p).any()}

# --- refit on everything for the model that actually answers queries ---
classifier = new_classifier()
classifier.fit(clf_data[CLF_FEATURES].to_numpy(float),
               clf_data['phase'].map(encode).to_numpy(int))

print(f'accuracy         {accuracy:.3f}   (chance {1/len(PHASES):.3f})')
print(f'within one phase {within_one:.3f}')
print(f'held-out rows    {len(truth):,}')
print()
for p, a in per_phase.items():
    print(f'  {p:16s} {a:.3f}')

## 5 · Phase-end model

Three quantile regressors on the phase's **total** length; the app subtracts elapsed weeks itself. Predicting "weeks remaining" directly asks the trees to learn a subtraction from a known input, which they approximate with a staircase of splits — it lost to the baseline by 63%.

Trained only on windows whose end was actually observed. In-progress phases have no target, and start-censored ones have a duration that is a lower bound.

In [ ]:
from xgboost import XGBRegressor

def new_regressor(q):
    return XGBRegressor(
        objective='reg:quantileerror', quantile_alpha=q,
        n_estimators=260, max_depth=3, learning_rate=0.07,
        subsample=0.9, colsample_bytree=0.9, min_child_weight=5,
        tree_method='hist', n_jobs=1, random_state=SEED, verbosity=0,
    )

dur_data = panel[
    panel['is_complete'] & ~panel['start_censored'] & panel['phase_total_weeks'].notna()
].dropna(subset=DUR_FEATURES)

# --- baseline: mean observed duration per phase, and which phases we may speak about ---
done = site_phases[site_phases['is_complete'] & ~site_phases['start_censored']]
mean_duration, trainable = {}, set()
for name, group in done.groupby('phase'):
    d = group['duration_weeks'].dropna()
    if len(d):
        mean_duration[name] = float(d.mean())
    if len(d) >= META['min_completed_phases']:
        trainable.add(name)

# --- out-of-fold error and interval coverage ---
rows = []
for train, test in window_folds(dur_data):
    if train.empty or test.empty:
        continue
    fitted = {}
    for q in QUANTILES:
        m = new_regressor(q)
        m.fit(train[DUR_FEATURES].to_numpy(float),
              train['phase_total_weeks'].to_numpy(float))
        fitted[q] = m.predict(test[DUR_FEATURES].to_numpy(float))
    elapsed = test['weeks_elapsed_in_phase'].to_numpy(float)
    # Predict the total, subtract elapsed, then clip: a phase cannot have a
    # negative number of weeks left to run.
    p10, p50, p90 = (np.clip(fitted[q] - elapsed, 0.0, None) for q in QUANTILES)
    typical = test['phase'].map(mean_duration).astype(float)
    typical = typical.fillna(np.mean(list(mean_duration.values())))
    rows.append(pd.DataFrame({
        'truth': test['weeks_remaining'].to_numpy(float),
        'low': np.minimum(p10, p90),      # thin data lets quantiles cross
        'mid': p50,
        'high': np.maximum(p10, p90),
        'phase': test['phase'].to_numpy(),
        'baseline': np.clip(typical - elapsed, 0.0, None),
    }))

oof = pd.concat(rows, ignore_index=True)
mae = float(np.abs(oof['mid'] - oof['truth']).mean())
baseline_mae = float(np.abs(oof['baseline'] - oof['truth']).mean())
raw_coverage = float(((oof['truth'] >= oof['low']) & (oof['truth'] <= oof['high'])).mean())
pad = conformal_pad(oof['truth'].to_numpy(), oof['low'].to_numpy(), oof['high'].to_numpy())
padded_low = np.clip(oof['low'] - pad, 0.0, None)
coverage = float(((oof['truth'] >= padded_low) & (oof['truth'] <= oof['high'] + pad)).mean())

# --- refit on everything ---
end_models = {}
for q in QUANTILES:
    m = new_regressor(q)
    m.fit(dur_data[DUR_FEATURES].to_numpy(float),
          dur_data['phase_total_weeks'].to_numpy(float))
    end_models[q] = m

print(f'MAE              {mae:.2f} weeks')
print(f'baseline MAE     {baseline_mae:.2f} weeks')
print(f'skill            {1 - mae/baseline_mae:+.1%}')
print(f'coverage @{LEVEL:.0%}     {coverage:.3f}   (raw {raw_coverage:.3f}, pad {pad:.2f} w)')
print(f'training rows    {len(dur_data):,}')
print(f'refuses          {sorted(set(PHASES) - trainable)}  -> insufficient_data')

## 6 · Save

`interval_pad` is **not optional**. Without it the band is the raw fitted quantiles, coverage falls to ~0.40, and nothing looks broken — so the app rejects a `phase_end.pkl` that is missing it.

In [ ]:
import joblib

FP = META['fingerprint']

joblib.dump({
    'schema_version': 1,
    'kind': 'phase_classifier',
    'fingerprint': FP,
    'feature_columns': CLF_FEATURES,
    'classes_': PHASES,
    'model': classifier,
    'scores': {
        'accuracy': accuracy,
        'n_train': int(len(clf_data)),
        'n_test': int(len(truth)),
        'n_test_windows': int(clf_data['window_id'].nunique()),
        'within_one_phase': within_one,
        'per_phase_accuracy': per_phase,
        'confusion': {},
        'feature_importance': dict(sorted(
            zip(CLF_FEATURES, map(float, classifier.feature_importances_)),
            key=lambda kv: kv[1], reverse=True)),
    },
}, 'phase_classifier.pkl')

joblib.dump({
    'schema_version': 1,
    'kind': 'phase_end',
    'fingerprint': FP,
    'feature_columns': DUR_FEATURES,
    'models': end_models,
    'interval_pad': pad,
    'mean_duration': mean_duration,
    'trainable_phases': sorted(trainable),
    'scores': {
        'mae_weeks': mae,
        'baseline_mae_weeks': baseline_mae,
        'n_train': int(len(dur_data)),
        'n_test': int(len(oof)),
        'n_test_windows': int(dur_data['window_id'].nunique()),
        'coverage': coverage,
        'raw_coverage': raw_coverage,
        'interval_pad_weeks': pad,
        'mae_by_phase': {
            p: float(np.abs(g['mid'] - g['truth']).mean())
            for p, g in oof.groupby('phase')
        },
        'feature_importance': dict(sorted(
            zip(DUR_FEATURES, map(float, end_models[0.50].feature_importances_)),
            key=lambda kv: kv[1], reverse=True)),
    },
}, 'phase_end.pkl')

print('wrote phase_classifier.pkl and phase_end.pkl')
print('fingerprint', FP)

## 7 · Download

Put both files in `models/` in the repo and restart the backend.

Confirm they were accepted — `dataset_fingerprint` must equal both models' fingerprint, otherwise the app logs a mismatch and retrains locally:

```bash
curl -s localhost:8000/api/health
```

In [ ]:
from google.colab import files
files.download('phase_classifier.pkl')
files.download('phase_end.pkl')